In [ ]:
from pyspark.sql import DataFrame, functions as F
import mlflow
import mlflow.sklearn

from transforms.calibration import (
    to_long_format,
    fit_all_calibration_curves,
    calibrate_matches,
    evaluate_calibration,
)

SILVER_TABLE = "stryktipset.silver.matches"
GOLD_TABLE = "stryktipset.gold.fact_match"  # built by 04_gold_fact_match
MLFLOW_EXPERIMENT = "/Users/adamwillner2@gmail.com/stryktipset-calibration"  # workspace experiment -- the implicit notebook one is temporary when the job runs from Git


In [ ]:
def load_matches(table: str = SILVER_TABLE) -> DataFrame:
    """Load the match dataset written by 02_silver_transform."""
    return spark.table(table)

In [ ]:
def merge_into_fact_match(df_calibrated: DataFrame, table: str = GOLD_TABLE) -> None:
    """Merge calibrated_1/calibrated_x/calibrated_2 into the existing
    fact_match table on match_id. Uses MERGE ... WITH SCHEMA EVOLUTION
    (Delta's SQL syntax for letting a merge add new columns) instead of
    the spark.databricks.delta.schema.autoMerge.enabled session config --
    that config isn't available on Databricks serverless.
    """
    source_view = "calibration_source"
    df_calibrated.select(
        "match_id", "calibrated_1", "calibrated_x", "calibrated_2"
    ).createOrReplaceTempView(source_view)

    spark.sql(f"""
        MERGE WITH SCHEMA EVOLUTION INTO {table} AS target
        USING {source_view} AS source
        ON target.match_key = source.match_id
        WHEN MATCHED THEN UPDATE SET
            target.calibrated_1 = source.calibrated_1,
            target.calibrated_x = source.calibrated_x,
            target.calibrated_2 = source.calibrated_2
    """)

In [ ]:
def main():
    """Load matches from silver, fit calibration curves on the full
    historical set, apply them to every match, merge the result into
    fact_match, and log the run to MLflow.

    No .cache()/.unpersist() -- Databricks serverless (Spark Connect)
    doesn't support PERSIST TABLE, so df_calibrated is recomputed for
    each step that needs it instead of being cached in memory.
    """
    df = load_matches()
    df_long = to_long_format(df)

    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    with mlflow.start_run(run_name="calibration_v1"):
        models = fit_all_calibration_curves(df_long)
        df_calibrated = calibrate_matches(df, models)

        mlflow.log_param("training_rows", df.count())
        metrics = evaluate_calibration(df_calibrated)
        mlflow.log_metrics(metrics)
        for outcome, model in models.items():
            mlflow.sklearn.log_model(model, f"isotonic_{outcome}")

        merge_into_fact_match(df_calibrated)
        print(f"Merged calibrated probabilities into {GOLD_TABLE}")

main()